# Анализ лояльности пользователей Яндекс Афиши

In [ ]:
#Установка зависимостей
!pip install sqlalchemy psycopg2-binary python-dotenv
!pip install pandas seaborn

### 1. Загрузка данных и их предобработка

**Задача 1.1:** Требуется написать SQL-запрос, выгружающий в датафрейм pandas необходимые данные из базы данных `data-analyst-afisha`:

- `user_id` — уникальный идентификатор пользователя, совершившего заказ;
- `device_type_canonical` — тип устройства, с которого был оформлен заказ (`mobile` — мобильные устройства, `desktop` — стационарные);
- `order_id` — уникальный идентификатор заказа;
- `order_dt` — дата создания заказа;
- `order_ts` — дата и время создания заказа;
- `currency_code` — валюта оплаты;
- `revenue` — выручка от заказа;
- `tickets_count` — количество купленных билетов;
- `days_since_prev` — количество дней от предыдущей покупки пользователя, для пользователей с одной покупкой — значение пропущено;
- `event_id` — уникальный идентификатор мероприятия;
- `service_name` — название билетного оператора;
- `event_type_main` — основной тип мероприятия (театральная постановка, концерт и так далее);
- `region_name` — название региона, в котором прошло мероприятие;
- `city_name` — название города, в котором прошло мероприятие.

In [ ]:
# Импорты
from dotenv import load_dotenv
import os  
import pandas as pd
from sqlalchemy import create_engine

In [ ]:
# Чтение .env файла
load_dotenv(dotenv_path='.env')

In [ ]:
# Подключение к бд
engine = create_engine(os.getenv('DB_CONNECTION_STRING')) 

In [ ]:
# Получение данных
query = '''
SELECT
  user_id,
  device_type_canonical,
  order_id,
  created_dt_msk AS order_dt,
  created_ts_msk       AS order_ts,
  currency_code,
  revenue,
  tickets_count,
  created_dt_msk::date - LAG(created_dt_msk::date) OVER (
    PARTITION BY user_id
    ORDER BY created_dt_msk
  ) AS days_since_prev,
  event_id,
  service_name,
  e.event_name_code AS event_name,
  event_type_main,
  r.region_name,
  c.city_name
FROM afisha.purchases
LEFT JOIN afisha.events e USING (event_id) 
LEFT JOIN afisha.city c USING (city_id)
LEFT JOIN afisha.regions r USING (region_id)
WHERE device_type_canonical IN ('mobile', 'desktop') 
  AND event_type_main != 'фильм'
ORDER BY user_id
'''
df = pd.read_sql_query(query, con=engine)
df